# State sync: Laminar to traitlets and back

The last piece. `own-bundle.ipynb` proved our Scala.js runs in the webview, but
its `render` ignored `model` on purpose. This one connects the two halves.

`Params` is defined **once**, in `example/src`, and compiled twice — into
`example.jvm` for the kernel and `example.js` for the browser. Neither end has a
hand-written JSON mapping. That is the argument for doing this in Scala at all.

```
./mill example.js.bundle
./mill example.jvm.publishLocal
```

In [10]:
// Pulls wedgie-kernel in transitively.
import $ivy.`io.github.quafadas::wedgie-example:0.1.0-SNAPSHOT`

import wedgie.EsmSource
import wedgie.example.Params
import wedgie.kernel.Widget

import $ivy.$                                                  


import wedgie.EsmSource

import wedgie.example.Params

import wedgie.kernel.Widget


## Load the bundle

Use `example.js.bundle`, not `fullLinkJS`. That task runs the linked output
through esbuild, and the difference is not cosmetic:

| | |
| --- | --- |
| `fullLinkJS` alone | 1414 KB — **the widget never renders**, silently |
| through esbuild | 549 KB — works |

`ModuleKind.ESModule` forfeits Closure and `scalaJSMinify` does not recover it,
so upickle's derivation machinery stays unminified. `_esm` is synced model state,
so the bundle travels inside `comm_open`, and 1.4 MB does not survive the trip.
Nothing is reported anywhere — the cells run clean and no widget appears.

`EsmSource.CommDelivered` would remove the `.ipynb` cost entirely, at the price
of probes 4 and 5.

**Do not save this notebook** with the widget output in it.


In [11]:
val bundlePath = java.nio.file.Paths.get(
  sys.props("user.home"), "Code", "wedgie",
  "out", "example", "js", "bundle.dest", "wedgie.js"
)

val bundle = java.nio.file.Files.readString(bundlePath)
println(s"bundle: ${bundle.length / 1024} KB")

bundle: 1413 KB


bundlePath: Path = /Users/simon/Code/wedgie/out/example/js/fullLinkJS.dest/main.js
bundle: String = """'use strict';
var $p;
var $fileLevelThis = this;
var $getOwnPropertyDescriptors = (Object.getOwnPropertyDescriptors || (() => {
  var ownKeysFun;
  if ((((typeof Reflect) !== "undefined") && Reflect.ownKeys)) {
    ownKeysFun = Reflect.ownKeys;
  } else {
    var getOwnPropertySymbols = (Object.getOwnPropertySymbols || ((o) => []));
    ownKeysFun = ((o) => Object.getOwnPropertyNames(o).concat(getOwnPropertySymbols(o)));
  }
  return ((o) => {
    var ownKeys = ownKeysFun(o);
    var descriptors = ({});
    var len = (ownKeys.length | 0);
    var i = 0;
    while ((i !== len)) {
      var key = ownKeys[i];
      Object.defineProperty(descriptors, key, ({
        "configurable": true,
        "enumerable": true,
        "writable": true,
        "value": Object.getOwnPropertyDescriptor(o, key)
      }));
      i = ((i + 1) | 0);
    }
    return descriptors;
  });
})());
function $Char

## The widget

Three controls, one shared state type. The initial value here is the kernel's;
the frontend reads every key off the model rather than using its own defaults.

In [12]:
val params = Widget(Params(n = 25, label = "runs", enabled = true), EsmSource.Inline(bundle))

params: Widget[Params] = wedgie.kernel.Widget@4d8c2550

## Direction 1 — browser to kernel

Move the slider, type in the box, toggle the checkbox. Then run this.

No buffer to inspect and no manual decoding: the inbound `update` messages have
already been merged into a `Params`.

In [19]:
params.state

res19: Params = Params(n = 57, label = "pushed from Scala", enabled = false)

## Direction 2 — kernel to browser

The controls should move without being touched. `set` diffs against current
state, so this puts only the changed keys on the wire.

In [14]:
params.set(Params(n = 80, label = "pushed from Scala", enabled = false))

res14: Params = Params(n = 80, label = "pushed from Scala", enabled = false)

In [15]:
// `modify` is the read-modify-write form.
params.modify(p => p.copy(n = math.min(100, p.n + 10)))

res15: Params = Params(n = 90, label = "pushed from Scala", enabled = false)

## The thing this is all for

A control driving a recalculation. Move the slider after running this, then
inspect the log.

`fromFrontend` matters: without it, an observer that calls `set` would drive
itself in a loop.

In [16]:
val results = scala.collection.mutable.ArrayBuffer.empty[String]

def recompute(p: Params): String =
  if !p.enabled then s"${p.label}: disabled"
  else s"${p.label}: sum(1..${p.n}) = ${(1 to p.n).sum}"

params.onChange { change =>
  if change.fromFrontend then
    results.synchronized { results += recompute(change.state) }
}

results: ArrayBuffer[String] = ArrayBuffer()
defined function recompute

In [17]:
// Move the slider a few times first.
results.synchronized(results.toList).takeRight(10).foreach(println)

## No feedback loop

Both ends refuse to echo, by the same rule: diff against what the other side
already holds, and send nothing when the diff is empty.

- **Kernel**: an inbound `update` merges into state and is never sent back.
- **Frontend**: `Bridge` diffs against the model's *current* attributes, so a
  change that arrived from the kernel produces an empty diff and no `save_changes`.

Neither needs a mutex or a suppression flag. If a loop ever appears, this is the
invariant that broke.

Check it: the count below should not grow while you are not touching the widget.

In [18]:
results.synchronized(results.size)

res18: Int = 0

## What is left

- `Entry`/`smoke` stays as a bisection tool: it mounts Laminar without a model,
  so "Laminar broke" and "sync broke" remain separable.
- Bundle delivery is now the open question, and it is about size, not
  feasibility. See `probes.ipynb` if you want more than one widget per notebook.